# 🌸 ShiroLLM - AI Chatbot (Google Colab Edition)
Jalankan AI Shiro menggunakan GPU gratis di Google Colab (T4 / A100)!

### Fitur Utama:
- 🚀 **Akselerasi GPU CUDA Penuh**: Respon super cepat (< 1 detik)
- 🧠 **Model Tercanggih**: Lumimaid-v0.2-8B (Llama-3 fine-tune khusus roleplay & percakapan karakter)
- 💡 **Self-Learning Continuous Memory**: Shiro belajar dan mengingat fakta, janji, emosi, dan preferensi dari setiap percakapan
- 🌐 **Web UI Publik via Ngrok**: Akses langsung antarmuka chat cantik dari HP atau browser PC Anda!

## Langkah 1: Cek GPU

In [ ]:
!nvidia-smi

## Langkah 2: Clone Repository

In [ ]:
%cd /content
!rm -rf ShiroLLM-AI-CHAT
!git clone https://github.com/Renn-devBI/ShiroLLM-AI-CHAT.git
%cd /content/ShiroLLM-AI-CHAT

## Langkah 3: Install Dependencies & llama-cpp-python dengan CUDA

In [ ]:
# Install dependencies dasar & pyngrok
!pip install -r requirements.txt
!pip install pyngrok flask-cors huggingface-hub

# Install llama-cpp-python dengan akselerasi CUDA untuk GPU Colab
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir

## Langkah 4: Download Model Tercanggih (Lumimaid 8B)

In [ ]:
!python download_model.py

## Langkah 5: Jalankan Server Shiro dengan Ngrok (Direkomendasikan! ⭐)
> **Dapatkan Authtoken Ngrok gratis**: Buka [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken), copy token Anda, lalu masukkan di bawah ini.

In [ ]:
import os
from pyngrok import ngrok

# Masukkan token ngrok Anda di sini (atau biarkan kosong untuk input interaktif)
NGROK_AUTHTOKEN = ""  # <-- Masukkan token Anda di sini jika ingin otomatis

if not NGROK_AUTHTOKEN:
    import getpass
    NGROK_AUTHTOKEN = getpass.getpass("Masukkan Ngrok Authtoken Anda (dari dashboard.ngrok.com): ")

# Setup ngrok token & buka tunnel ke port 7474
ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
ngrok.kill() # Tutup koneksi lama jika ada
public_url = ngrok.connect(7474)

print("\n" + "="*60)
print(f"🎉 LINK WEB UI SHIRO (NGROK): {public_url}")
print("="*60 + "\n")

# Set GPU layers ke -1 (offload semua layer model ke GPU VRAM Colab)
os.environ["N_GPU_LAYERS"] = "-1"

# Jalankan server web Shiro
!python web.py

## Alternatif (Tanpa Token Ngrok): Jalankan dengan Cloudflare Tunnel
Gunakan cell ini jika Anda tidak memiliki akun ngrok.

In [ ]:
import os
import subprocess
import threading
import time
import re

# Set GPU layers ke -1
os.environ["N_GPU_LAYERS"] = "-1"

# Download Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

def start_cloudflare_tunnel():
    time.sleep(3)
    p = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:7474"], 
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
        if match:
            print("\n" + "="*60)
            print(f"👉 LINK WEB UI ANDA: {match.group(1)}")
            print("="*60 + "\n")
            break

threading.Thread(target=start_cloudflare_tunnel, daemon=True).start()

# Jalankan server web Shiro
!python web.py

## (Opsional) Sinkronisasi / Backup Memori ke Google Drive
Jalankan cell ini jika ingin menyimpan ingatan Shiro ke Google Drive agar tidak hilang saat runtime Colab di-restart.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')
backup_dir = '/content/drive/MyDrive/Shiro_Memory_Backup'
os.makedirs(backup_dir, exist_ok=True)

# Backup file ingatan saat ini ke Google Drive
if os.path.exists('ingatan_shiro.json'):
    shutil.copy('ingatan_shiro.json', os.path.join(backup_dir, 'ingatan_shiro.json'))
    print(f"✓ Ingatan Shiro berhasil dibackup ke: {backup_dir}")
